In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

GPKG = "data/catastro_manzana.gpkg"
JSON_FILE = "data/lod1_meshes.json"
BLEND_FILE = "blender/lod1.blend"
GLB_FILE = "blender/lod1.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"

# Local origin of the Blender scene, in EPSG:25830 meters (x, y, z):
ORIGIN = (440000.0, 4473700.0, 640.0)

# Parcels of the Colegio Imperial (full detail); the rest of the block is context:
CONJUNTO = ["0139402VK4703G", "0139423VK4703G", "0139424VK4703G"]

In [ ]:
# Celda 2
import os
import json
import subprocess
import shapely
import geopandas as gpd
from shapely.geometry.polygon import orient

os.makedirs("blender", exist_ok=True)

In [ ]:
# Celda 3
# Building parts with heights; parts with 0 floors above ground are basements only, so we skip them:
parts = gpd.read_file(GPKG, layer="building_parts_alturas")
parts = parts[parts["numberOfFloorsAboveGround"] > 0].copy()
parts["z_top"] = parts["z_base"] + parts["height_m"]
print(len(parts), "parts above ground")

In [ ]:
# Celda 4
# Turn one footprint into a closed prism (vertices and faces), in local coordinates:
# Walls: one quad per edge of each ring. Top and bottom: triangles of the polygon (holes respected).
def prism(polygon, z_bottom, z_top):
    polygon = orient(polygon, 1.0)   # Outer ring counterclockwise, holes clockwise
    x0, y0, z0 = ORIGIN
    vertices = []
    faces = []

    # Walls:
    rings = [polygon.exterior] + list(polygon.interiors)
    for ring in rings:
        coords = list(ring.coords)[:-1]   # The last point repeats the first one
        n = len(coords)
        start = len(vertices)
        for x, y in coords:
            vertices.append((x - x0, y - y0, z_bottom - z0))
            vertices.append((x - x0, y - y0, z_top - z0))
        for i in range(n):
            j = (i + 1) % n
            a_low = start + 2 * i
            b_low = start + 2 * j
            faces.append((a_low, b_low, b_low + 1, a_low + 1))

    # Top (facing up) and bottom (facing down):
    triangles = shapely.constrained_delaunay_triangles(polygon)
    for triangle in triangles.geoms:
        triangle = orient(triangle, 1.0)
        corners = list(triangle.exterior.coords)[:3]
        start = len(vertices)
        for x, y in corners:
            vertices.append((x - x0, y - y0, z_top - z0))
        for x, y in corners:
            vertices.append((x - x0, y - y0, z_bottom - z0))
        faces.append((start, start + 1, start + 2))
        faces.append((start + 5, start + 4, start + 3))

    return vertices, faces

In [ ]:
# Celda 5
# Build every prism and save them all in one JSON file for Blender:
meshes = []
for i, row in parts.iterrows():
    vertices, faces = prism(row.geometry, row["z_base"], row["z_top"])
    if row["parcel"] in CONJUNTO:
        group = "conjunto"
    else:
        group = "contexto"
    meshes.append({
        "name": row["parcel"] + "_" + str(row["localId"]).split("_")[-1],
        "parcel": row["parcel"],
        "group": group,
        "height_m": float(row["height_m"]),
        "height_method": row["height_method"],
        "lod": 1,
        "vertices": vertices,
        "faces": faces,
    })

with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": meshes}, f)
print(len(meshes), "meshes ->", JSON_FILE)

In [ ]:
# Celda 6
# Run Blender in the background with the bpy script:
command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
result = subprocess.run(command, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if "Objects" in line or "Error" in line or "Saved" in line:
        print(line)
print("Return code:", result.returncode)
if result.returncode != 0:
    print(result.stderr[-2000:])

In [ ]:
# Celda 7
# Check the output files:
for path in [BLEND_FILE, GLB_FILE]:
    print(path, round(os.path.getsize(path) / 1e6, 2), "MB")